In [ ]:
import pandas as pd
import numpy as np
from langdetect import detect
from top2vec import Top2Vec

# 1 Read-in dataframe

In [ ]:
# read in dataframe, only use the columns: reviewText, summary, overall, asin
df = pd.read_csv('apple.csv', usecols=["reviewText", "summary", "overall", "asin"])

In [ ]:
print(df.isna().sum())
print(df.info())

print(df["overall"].min())
print(df["overall"].max())

# 2 Replace na values and start ratings in reviewText

In [ ]:
# look at the na values in summary
print(df[df["summary"].isna()].shape)
df[df["summary"].isna()].head()

In [ ]:
# look at the "star rating" text in the summary column
stars = ["One Star", "Two Stars", "Three Stars", "Four Stars", "Five Stars", "1 Star", "2 Stars", "3 Stars", "4 Stars", "5 Stars"]
df[df["summary"].isin(stars)]

In [ ]:
# fill na values in summary with empty string
print("Fill {0} na values with empty string".format(df[df["summary"].isna()].shape[0]))
df.loc[df["summary"].isna(), "summary"] = ""

In [ ]:
# fill values in summary with only a star rating
print("Fill {0} na values with empty string".format(df[df["summary"].isin(stars)].shape[0]))
df.loc[df["summary"].isin(stars), "summary"] = ""
print("\nDoblecheck:")
indexes = [31, 32, 33, 34, 35, 132, 133, 134, 135, 136]
df.loc[indexes, :]

# 3 Replace na values in reviewText

In [ ]:
# look at the na values in reviewText
print("\nLOOK AT THE NA VALUES IN summary")
print(df[df["reviewText"].isna()].shape)
df[df["reviewText"].isna()].head()

In [ ]:
# fill na values in reviewText with empty string
print("Fill {0} na values with empty string".format(df[df["reviewText"].isna()].shape[0]))
df.loc[df["reviewText"].isna(), "reviewText"] = ""
print("\nDoblecheck:")
indexes = [3541, 3772, 4534, 4560, 4695]
df.loc[indexes, :]

In [ ]:
print("There are {0} rows with empty reviewText and non-empty summary".format(df.loc[(df["reviewText"] == "") & (df["summary"] != "")].shape[0]))
df.loc[(df["reviewText"] == "") & (df["summary"] != "")]

# 4 Combine reviewText and summary into corpus and clean

In [ ]:
# combine the reviewText and summary columns
df["corpus"] = df["summary"] + " " + df["reviewText"]
df = df.drop(columns=["reviewText", "summary"])
df.head()

In [ ]:
# drop rows with empty corpus
print("There are {0} rows with empty corpus".format(df[df["corpus"] == " "].shape[0]))
df = df[df["corpus"] != " "]

In [ ]:
# drop rows that solely contain non-english characters
print("There are {0} rows with only non-english characters".format(df[~df["corpus"].str.contains("[a-zA-Z]")].shape[0]))
df = df[df["corpus"].str.contains("[a-zA-Z]")]

In [ ]:
# drop rows that contain less than 10 words

min_size = 5

print("There are {0} rows with less than {1} words".format(df[df["corpus"].str.split().str.len() < min_size].shape[0], min_size))
print("\nRemoving rows such as:")
print(df[df["corpus"].str.split().str.len() < min_size].head(5))
df = df[df["corpus"].str.split().str.len() >= min_size]
print(f"\nNumber of rows after filtering: {df.shape[0]}")

In [ ]:
# replace \n with space
print("There are {0} rows where corpus contains '\\n'".format(df[df["corpus"].str.contains("\n")].shape[0]))
df["corpus"] = df["corpus"].str.replace("\n", " ")

In [ ]:
add = []
counter = 0
for row in df["corpus"]:
    try:
        if detect(row) != "en":
            add.append(row)
            counter += 1
    except:
        print("This row throws and error:", row)
print(add)
print(counter)

In [ ]:
df.to_csv("apple_preprocessed.csv", index=False)